# Adivina el mineral
Ejecuta las dos celdas en orden. Aparecerá una foto y una caja para escribir.
Pulsa **Comprobar** y después **Siguiente mineral**. No repite fotos durante la partida.
Mantén la sesión de Colab conectada mientras juegas. No necesitas escribir rutas ni subir fotos.

In [ ]:
# Ejecuta esta celda primero. Descarga las fotos automáticamente.
%pip -q install Pillow ipywidgets

from pathlib import Path
import subprocess

# Al abrir un cuaderno de GitHub, Colab no descarga el resto del repositorio.
base = Path('/content') if Path('/content').is_dir() else Path.cwd()
candidatos = [Path.cwd() / 'minerales', base / 'minerales-juego' / 'minerales']
directorio = next((p for p in candidatos if p.is_dir() and any(p.glob('*.png'))), None)
if directorio is None:
    destino = base / 'minerales-juego-colab'
    if not destino.exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/mvbuenodelgado/minerales-juego.git',
                        str(destino)], check=True)
    directorio = destino / 'minerales'
if not directorio.is_dir():
    raise RuntimeError('No se han podido descargar las fotos. Vuelve a ejecutar esta celda.')
print('Fotos listas. Ahora ejecuta la siguiente celda.')


In [ ]:
from pathlib import Path
from io import BytesIO
import random
import unicodedata
import html
import ipywidgets as widgets
from PIL import Image, ImageOps
from IPython.display import display, clear_output


def normalizar(texto):
    texto = unicodedata.normalize('NFD', texto.strip().casefold())
    return ' '.join(''.join(c for c in texto if unicodedata.category(c) != 'Mn').split())


class JuegoMinerales:
    def __init__(self, directorio):
        self.archivos = sorted(p for p in Path(directorio).iterdir()
                               if p.suffix.lower() in ('.png', '.jpg', '.jpeg'))
        if not self.archivos:
            raise ValueError('No hay fotos en la carpeta de minerales.')
        self.foto = widgets.Image(format='png', layout=widgets.Layout(
            max_width='100%', max_height='320px', object_fit='contain'))
        self.respuesta = widgets.Text(placeholder='Escribe aquí el nombre del mineral',
            layout=widgets.Layout(width='100%', max_width='520px'))
        self.comprobar = widgets.Button(description='Comprobar', button_style='primary')
        self.siguiente = widgets.Button(description='Siguiente mineral', disabled=True)
        self.reiniciar = widgets.Button(description='Empezar de nuevo')
        self.mensaje = widgets.HTML()
        self.marcador = widgets.HTML()
        self.panel = widgets.VBox([
            widgets.HTML('<h2>🔎 Adivina el mineral</h2>'),
            self.marcador, self.foto,
            widgets.HTML('<b>¿Cómo se llama este mineral?</b>'),
            self.respuesta,
            widgets.HBox([self.comprobar, self.siguiente],
                         layout=widgets.Layout(flex_flow='row wrap')),
            self.mensaje, self.reiniciar
        ], layout=widgets.Layout(width='100%', max_width='600px', padding='12px'))
        self.comprobar.on_click(self.evaluar)
        self.siguiente.on_click(self.avanzar)
        self.reiniciar.on_click(self.nueva_partida)
        self.nueva_partida()

    def nueva_partida(self, _=None):
        self.orden = random.sample(self.archivos, len(self.archivos))
        self.indice = 0
        self.aciertos = 0
        self.mostrar()

    def mostrar(self):
        self.actual = self.orden[self.indice]
        with Image.open(self.actual) as original:
            imagen = ImageOps.exif_transpose(original).convert('RGB')
            imagen.thumbnail((700, 320))
            datos = BytesIO()
            imagen.save(datos, format='PNG')
        self.foto.value = datos.getvalue()
        self.foto.layout.display = ''
        self.respuesta.value = ''
        self.respuesta.disabled = False
        self.comprobar.disabled = False
        self.siguiente.disabled = True
        self.mensaje.value = ''
        self.marcador.value = (f'Mineral {self.indice + 1} de {len(self.orden)}'
                              f' · Aciertos: {self.aciertos}')

    def evaluar(self, _=None):
        if self.comprobar.disabled:
            return
        if not self.respuesta.value.strip():
            self.mensaje.value = '✍️ Escribe un nombre antes de comprobar.'
            return
        nombre = self.actual.stem
        if normalizar(self.respuesta.value) == normalizar(nombre):
            self.aciertos += 1
            self.mensaje.value = '<b style="color:green">✅ ¡Correcto!</b>'
        else:
            self.mensaje.value = f'❌ El nombre correcto es <b>{html.escape(nombre)}</b>.'
        self.respuesta.disabled = True
        self.comprobar.disabled = True
        self.siguiente.disabled = False
        self.marcador.value = (f'Mineral {self.indice + 1} de {len(self.orden)}'
                              f' · Aciertos: {self.aciertos}')

    def avanzar(self, _=None):
        if self.siguiente.disabled:
            return
        self.indice += 1
        if self.indice == len(self.orden):
            self.foto.layout.display = 'none'
            self.siguiente.disabled = True
            self.marcador.value = f'Partida terminada: {self.aciertos} / {len(self.orden)}'
            self.mensaje.value = '🏁 ¡Has repasado todos los minerales! Pulsa Empezar de nuevo para repetir.'
            return
        self.mostrar()


clear_output(wait=True)
juego = JuegoMinerales(directorio)
display(juego.panel)
